## საუკეთესო მოდელით პროგნოზი და Kaggle Submission

ეს notebook:
1. MLflow Model Registry-დან ჩამოტვირთავს საუკეთესო მოდელს
2. test.csv-ზე გაუშვებს პროგნოზს **იმავე** preprocessing-ით რაც experiment-ში გამოვიყენეთ
3. შექმნის submission.csv-ს Kaggle-ზე ასატვირთად

In [1]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
import dagshub
from sklearn.preprocessing import LabelEncoder

c:\Users\taso\ML\.venv\Lib\site-packages\mlflow\utils\requirements_utils.py:20: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources  # noqa: TID251


## ნაბიჯი 1 — MLflow-სთან დაკავშირება

In [2]:
dagshub.init(repo_owner="aband22", repo_name="house-prices", mlflow=True)
print("MLflow connected!")

Accessing as aband22

Initialized MLflow to track repo "aband22/house-prices"

Repository aband22/house-prices initialized!

MLflow connected!


## ნაბიჯი 2 — საუკეთესო მოდელის ჩამოტვირთვა Model Registry-დან

model_experiment.ipynb-ში დავარეგისტრირეთ `house-prices-best-model` სახელით.

In [3]:
model = mlflow.sklearn.load_model("models:/house-prices-best-model/4")
print("მოდელი ჩამოიტვირთა Model Registry-დან!")
print("მოდელის ტიპი:", type(model).__name__)

მოდელი ჩამოიტვირთა Model Registry-დან!
მოდელის ტიპი: GradientBoostingRegressor


## ნაბიჯი 3 — მონაცემების ჩატვირთვა და Preprocessing

**მნიშვნელოვანი:** ზუსტად იგივე ნაბიჯები როგორც experiment-ში:
- `clean_smart()` — Cleaning მიდგომა 3
- `add_features()` — TotalSF, HouseAge, QualxSF
- `encode_label()` — Label Encoding
- RF top 25 features — იგივე სია

In [4]:
test = pd.read_csv("data/test.csv")
print("Test shape:", test.shape)

Test shape: (1459, 80)


In [11]:
def clean_smart(df):
    df = df.copy()
    cols_to_drop = ['PoolQC', 'MiscFeature', 'Alley', 'Fence']
    df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])
    df['LotFrontage'] = df.groupby('Neighborhood')['LotFrontage'].transform(
        lambda x: x.fillna(x.median()))
    df['LotFrontage'] = df['LotFrontage'].fillna(df['LotFrontage'].median())
    num_cols = df.select_dtypes(include=[np.number]).columns
    for col in num_cols:
        df[col] = df[col].fillna(df[col].median())
    cat_cols = df.select_dtypes(include=['object']).columns
    for col in cat_cols:
        df[col] = df[col].fillna(df[col].mode()[0])
    return df

def add_features(df):
    df = df.copy()
    df['TotalSF'] = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']
    df['HouseAge'] = df['YrSold'] - df['YearBuilt']
    df['QualxSF'] = df['OverallQual'] * df['TotalSF']
    return df

# Label Encoding
def encode_label(df):
    df = df.copy()
    cat_cols = df.select_dtypes(include=['object']).columns
    for col in cat_cols:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))
    return df

# RF top 25 — experiment-ში შერჩეული features
top_rf_features = [
    'QualxSF', 'BsmtUnfSF', 'YearRemodAdd', 'GarageArea', 'MSZoning',
    'LotArea', 'HouseAge', 'BsmtFinSF1', 'CentralAir', 'GrLivArea',
    'OverallCond', 'TotalSF', '1stFlrSF', 'GarageCars', 'GarageFinish',
    'LotFrontage', 'YearBuilt', 'Neighborhood', 'SaleCondition', '2ndFlrSF',
    'TotalBsmtSF', 'OpenPorchSF', 'KitchenQual', 'GarageYrBlt', 'WoodDeckSF'
]

test_clean  = clean_smart(test)
test_feat   = add_features(test_clean)
test_enc    = encode_label(test_feat)
X_test      = test_enc[top_rf_features]

print("X_test shape:", X_test.shape)
print("Missing values:", X_test.isnull().sum().sum())

X_test shape: (1459, 25)
Missing values: 0


## პროგნოზი

მოდელი log(SalePrice)-ზე ისწავლა, ამიტომ `np.expm1()` გამოვიყენებთ ნამდვილი ფასის დასაბრუნებლად:

```
log(SalePrice) → predict → expm1() → SalePrice
```

In [12]:
predictions_log = model.predict(X_test)
predictions     = np.expm1(predictions_log)

print("პროგნოზები:")
print(f"  მინ:    ${predictions.min():,.0f}")
print(f"  მაქს:   ${predictions.max():,.0f}")
print(f"  საშუალო: ${predictions.mean():,.0f}")

პროგნოზები:
  მინ:    $39,787
  მაქს:   $613,917
  საშუალო: $177,122


## Submission

In [13]:
submission = pd.DataFrame({
    'Id': test['Id'],
    'SalePrice': predictions
})

submission.to_csv('submission.csv', index=False)

print("submission.csv შექმნილია!")
print(f"სტრიქონები: {len(submission)}")
print()
print(submission.head(10).to_string(index=False))

submission.csv შექმნილია!
სტრიქონები: 1459

  Id     SalePrice
1461 127364.597565
1462 167462.335767
1463 175426.119161
1464 188235.563780
1465 183586.306762
1466 175736.459083
1467 187822.885780
1468 169876.553835
1469 181733.172599
1470 120372.436415
